In [1]:
from itertools import chain
from pathlib import Path

from tqdm.notebook import tqdm

from IPython.display import Markdown, display
import subprocess

import ghapicache
import release_notes

%load_ext autoreload
%autoreload 1
%aimport githelpers
%aimport ghapicache
%aimport release_notes

In [ ]:
import githelpers

# Reload token
cached = ghapicache.GhApiCache(token=ghapicache._load_ghapi_token(), repo="g4vg")
githelpers.REPO = Path.home() / "Code/g4vg"
ghapi = cached.api

In [3]:
# Load teams
TEAMS = {}
for t in tqdm(cached.teams()):
    TEAMS[t["name"]] = release_notes.get_team(cached, t)

team_members = frozenset(chain(*(t.members for t in TEAMS.values())))
org_members = frozenset(m["login"] for m in cached.org_members())
TEAMS["affiliates"] = release_notes.Team(
    description="Associated with Celeritas but not core members",
    members=(org_members - team_members),
)

# Initialize the user cache
local_repo = Path("/Users/seth/Code/celeritas-temp")
user_cache = release_notes.UserCache(cached, local_repo / "scripts/release/users.json")
for login in tqdm(org_members):
    user_cache[login]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

# Draft github release

In [15]:
release_md = release_notes.ReleaseMetadata(
    release="1.0.7", merge_bases=["v1.0.6"], target_branch="upstream/main"
)

In [16]:
release_notes.create_local_tag(release_md)

Updating release target from upstream/main to 1a6619c7e12841071308eeb1b635bc2be9d1624e


In [17]:
import tempfile

tgz_data = githelpers.git_archive_tgz(release_md.target_branch)
tgz_name = f"{cached.repo}-{release_md.release}.tar.gz"

with tempfile.TemporaryDirectory(delete=False) as tmpdir:
    with open(Path(tmpdir) / tgz_name, "wb") as f:
        f.write(tgz_data)
    subprocess.run(["open", tmpdir])

Decompress and smoke check the resulting gzip file, then:

In [7]:
subprocess.run(
    ["cmake", "-P", "cmake/CgvFindVersion.cmake"],
    cwd=Path(tmpdir) / f"{cached.repo}-{release_md.release}",
)

VERSION="1.0.7"
VERSION_STRING="1.0.7"


CompletedProcess(args=['cmake', '-P', 'cmake/CgvFindVersion.cmake'], returncode=0)

In [13]:
release_md

ReleaseMetadata(release='1.0.7', merge_bases=['v1.0.6'], target_branch='1a6619c7e12841071308eeb1b635bc2be9d1624e')

In [18]:
ghapi = cached.api

gh_release = release_notes.find_release(ghapi, release_md.release)
if not gh_release:
    gh_release = release_notes.create_release(ghapi, release_md, "")
    tgz_asset = release_notes.upload_asset(cached, gh_release, tgz_data, tgz_name)

No release found for version: 1.0.7
Draft release 1.0.7 created: https://github.com/celeritas-project/g4vg/releases/tag/untagged-6201159970f90ae9c66e
Uploaded artifact: https://github.com/celeritas-project/g4vg/releases/download/untagged-6201159970f90ae9c66e/g4vg-1.0.7.tar.gz


In [19]:
sha256_hash = release_notes.prepare_spack_update("g4vg", release_md, tgz_data)

Spack version copied to clipboard!


In [20]:
display(
    Markdown(f"""
## Time to publish!

1. Visit [the github release page]({gh_release["html_url"]})
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is {sha256_hash}
4. Double check the name and release notes
5. Edit and publish!
""")
)


## Time to publish!

1. Visit [the github release page](https://github.com/celeritas-project/g4vg/releases/tag/untagged-6201159970f90ae9c66e)
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is fb70f509cc85b0d5311b70246e141ffdd47abf9b175cf95aa0fd1b75edae616e
4. Double check the name and release notes
5. Edit and publish!


# Push to zenodo

In [ ]:
import zenodoapi

%aimport zenodoapi

In [17]:
# Load the Zenodo token
token_path = Path.home() / ".config/zenodo-token"
with open(token_path) as f:
    zenodo_token = f.read().strip()

zenodo = zenodoapi.Zenodo(zenodo_token)
zenodo.api_url = "https://zenodo.org/api/"

In [ ]:
def load_contributions(release_md):
    prs = release_notes.PullRequestRange(release_md)
    count_contrib = release_notes.ContributionCounter(cached)
    for pr in tqdm(prs.pull_ids):
        count_contrib(pr)

    # Create author list
    return count_contrib.sorted()


def download_tarball(gh_release):
    asset = gh_release["assets"][0]
    name = asset["name"]
    content = cached.download_file(asset["url"], ext=Path(name).suffix)
    assert content is not None
    return (content, name)


make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=user_cache, teams=TEAMS)

## Minor release

This assumes the major release has already been published upstream into Zenodo.

In [ ]:
assert not release_md.is_major()
major, minor, patch = release_md.as_version()
release_md = release_notes.ReleaseMetadata.from_comprehensive_version(
    major, minor, patch
)
major_dep = zenodo.find_deposition(f"G4VG {major}.{minor}")

In [29]:
release_md

ReleaseMetadata(release='1.0.6', merge_bases=['v1.0.0-dev^'], target_branch='v1.0.6')

In [30]:
zenodo.get_deposition(major_dep.get_latest_version().id)

ZenodoDeposition(id=16949066, title="G4VG 1.0")

In [32]:
gh_release = release_notes.find_release(ghapi, release_md.release)

In [ ]:
# Get the Zenodo metadata
from requests import HTTPError


old_md = major_dep.data["metadata"]
new_md = make_zenodo_md(load_contributions(release_md), release_md, gh_release)
# Only update editors, not team members
old_contrib = new_md["contributors"]
new_contrib = [u for u in old_contrib if u["type"] == "Editor"]
old_contrib = [u for u in old_contrib if u["type"] != "Editor"]
new_md["contributors"] = new_contrib + old_contrib
# Don't change the title
new_md["title"] = major_dep.data["metadata"]["title"]
# Add the body
new_md["description"] = "\n\n".join([
    old_md["description"],
    f"<h1>Version {release_md.release}</h1>",
    new_md["description"],
])
try:
    new_vers = major_dep.create_new_version()
except HTTPError as e:
    # Maybe the version already exists
    if e.response.status_code == 400:
        print("Draft may already exist:", e)
        new_vers = major_dep.get_latest_draft()
    else:
        raise
try:
    new_vers.update(new_md)
except HTTPError as e:
    # Something else went wrong? Try to upload the tarball
    print("Draft may already exist:", e)

# Upload the release
(content, name) = download_tarball(gh_release)
new_vers.upload(content, name)
# The old tarball may still be there (this is buggy) so delete it
new_vers.refresh()
for file in new_vers.get_files():
    if file.filename != name:
        try:
            file.delete()
        except Exception as e:
            print(f"Failed to delete {file.filename}: {e}")
new_vers.refresh()

  0%|          | 0/38 [00:00<?, ?it/s]

Created new version 18273294: https://zenodo.org/deposit/18273294
Updated deposition at https://zenodo.org/deposit/18273294 : G4VG 1.0
Uploaded g4vg-1.0.6.tar.gz: version cbbad57e-4aa4-4f6f-a324-d3ebf61058ea
Failed to delete g4vg-1.0.5.tar.gz: Expecting value: line 1 column 1 (char 0)


ZenodoDeposition(id=18273294, title="G4VG 1.0")

## Major release

In [57]:
assert release_md.is_major()
major, minor, patch = release_md.as_version()
release_md = release_notes.ReleaseMetadata.from_comprehensive_version(minor)
title = f"Celeritas {major}.{minor}"

In [58]:
zmd = make_zenodo_md(load_contributions(release_md), release_md, gh_release)
zmd["title"] = title

# Create the Zenodo deposition
deposition = zenodo.create_deposition(zmd)

  0%|          | 0/234 [00:00<?, ?it/s]

Created deposition 15281110 at https://zenodo.org/deposit/15281110 : Celeritas 0.6


In [60]:
# Upload the release tarball
deposition.upload(artifact_tgz, f"celeritas-{release_md.release}.tar.gz")

print(deposition.html)
print("""
IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses
""")

Uploaded celeritas-0.6.0.tar.gz: version bbd63016-30a1-4be9-9a74-329ca975bed9
https://zenodo.org/deposit/15281110

IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses

